# Movie Recommendation System: Modeling & Evaluation

This notebook demonstrates and evaluates three movie recommendation strategies on MovieLens 100K alongside a random floor baseline:
- **Random (floor):** uniform random sample of unrated movies
- **Content-based:** cosine similarity across 19 binary genre indicators with rating-count tie-breaking
- **Popularity:** share of positive ratings (`rating >= 4`) among movies with at least `min_ratings` ratings
- **Hybrid:** min-max scaled weighted blend `alpha * content_score + (1 - alpha) * popularity_score`

### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_movielens
from src.content_based import ContentBasedRecommender
from src.popularity import PopularityRecommender
from src.hybrid import HybridRecommender
from src.random_baseline import RandomRecommender
from src.evaluation import (
    compare_models,
    compare_user_recommendations,
    evaluate_by_user_activity,
    train_test_split_by_time,
    user_genre_profile,
)

### Load Dataset

In [ ]:
ratings, movies, users = load_movielens()

## Content-Based Recommendations

Initialize `ContentBasedRecommender` and generate top-10 genre-similar recommendations for User 1.

In [ ]:
model = ContentBasedRecommender(movies)

In [ ]:
recommendations = model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(recommendations)

## Popularity-Based Recommendations

The baseline recommends movies with at least `min_ratings` ratings and the highest share of positive (>= threshold) ratings. Every user gets the same list, minus movies they've already rated.

In [ ]:
popularity_model = PopularityRecommender(movies, min_ratings=50)

popular_recommendations = popularity_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(popular_recommendations)

## Hybrid Recommendations

Combines both scores: `score = alpha * content_score + (1 - alpha) * popularity_score`. Both scores are rescaled to 0-1 first. `alpha=1` is pure content-based and `alpha=0` is pure popularity.

In [ ]:
hybrid_model = HybridRecommender(movies, alpha=0.5, min_ratings=50)

hybrid_recommendations = hybrid_model.recommend(
    ratings=ratings,
    user_id=1,
    n=10,
    threshold=4
)

display(hybrid_recommendations)

## Evaluation

Each user's most recent 20% of ratings are held out as a test set. Each model only sees the training ratings, and we score top-10 recommendations against movies the user rated 4 or higher:

- **precision@10**: share of the 10 recommendations the user liked
- **recall@10**: share of the user's liked held-out movies that were recommended
- **hit_rate@10**: share of users with at least one liked movie in their top 10
- **ndcg@10**: rank-aware normalized discounted cumulative gain (rewards placing hits near the top)
- **mrr@10**: mean reciprocal rank of the first liked recommendation
- **coverage**: share of the whole catalog recommended to at least one user
- **personalization**: average `1 - cosine_similarity` across users' top-10 recommendation lists (`0` = identical lists, `1` = disjoint)

The random recommender is a floor: any useful model should clearly beat it on accuracy metrics (while random spreads uniformly across the catalog).

In [ ]:
train, test = train_test_split_by_time(ratings, test_fraction=0.2)

print(f"Train ratings: {len(train):,}")
print(f"Test ratings:  {len(test):,}")
print(f"Users in split: {train['user_id'].nunique()}")

In [ ]:
models = {
    "Random (floor)": RandomRecommender(movies, seed=0),
    "Content-based": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    "Hybrid (alpha=0.5)": HybridRecommender(movies, alpha=0.5, min_ratings=50),
}

comparison = compare_models(models, train, test, k=10, threshold=4)
display(comparison.round(4))

In [ ]:
metrics = [
    "precision@10", "recall@10", "hit_rate@10",
    "ndcg@10", "mrr@10", "coverage", "personalization",
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, metric in zip(axes.flat, metrics):
    bars = comparison[metric].plot.bar(ax=ax, color=["gray", "steelblue", "darkorange", "seagreen"])
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)

axes.flat[-1].set_visible(False)
plt.suptitle("Recommender Comparison Across Accuracy, Ranking, and Diversity Metrics")
plt.tight_layout()
plt.show()

### Choosing alpha

Sweep `alpha` from `0.0` (pure popularity) to `1.0` (content-based with popularity tie-breaking) in steps of `0.1` to map the accuracy vs. personalization/coverage trade-off.

In [ ]:
alphas = [round(i * 0.1, 1) for i in range(11)]

sweep = compare_models(
    {a: HybridRecommender(movies, alpha=a, min_ratings=50) for a in alphas},
    train, test, k=10, threshold=4
)
sweep.index.name = "alpha"
display(sweep.round(4))

In [ ]:
sweep_metrics = ["precision@10", "recall@10", "hit_rate@10", "ndcg@10", "coverage", "personalization"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, metric in zip(axes.flat, sweep_metrics):
    ax.plot(sweep.index, sweep[metric], marker="o", linewidth=2)
    ax.set_title(metric)
    ax.set_xlabel("alpha")
    ax.grid(True, alpha=0.3)

plt.suptitle("Hybrid Performance by Alpha (0 = popularity, 1 = content-based)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(sweep["coverage"], sweep["precision@10"], marker="o", linestyle="-", color="indigo")
for alpha_val, row in sweep.iterrows():
    plt.annotate(f"{alpha_val:.1f}", (row["coverage"], row["precision@10"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
plt.title("Precision@10 vs. Catalog Coverage Trade-Off Across Alpha")
plt.xlabel("Catalog Coverage")
plt.ylabel("Precision@10")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_alpha = float(sweep["precision@10"].idxmax())
print(f"Best alpha by precision@10: {best_alpha}")
print(f"Hybrid at best alpha:  precision@10={sweep.loc[best_alpha, 'precision@10']:.4f}, ndcg@10={sweep.loc[best_alpha, 'ndcg@10']:.4f}")
print(f"Popularity (alpha=0):  precision@10={sweep.loc[0.0, 'precision@10']:.4f}, ndcg@10={sweep.loc[0.0, 'ndcg@10']:.4f}")
print(f"Content    (alpha=1):  precision@10={sweep.loc[1.0, 'precision@10']:.4f}, ndcg@10={sweep.loc[1.0, 'ndcg@10']:.4f}")

### Robustness check

Re-run the comparison across hold-out sizes `0.1`, `0.2`, and `0.3` to verify whether model rankings remain stable as training history shrinks.

In [ ]:
robustness_prec = {}
robustness_ndcg = {}
for fraction in [0.1, 0.2, 0.3]:
    tr, te = train_test_split_by_time(ratings, test_fraction=fraction)
    res = compare_models(
        {
            "Random (floor)": RandomRecommender(movies, seed=0),
            "Content-based": ContentBasedRecommender(movies),
            "Popularity": PopularityRecommender(movies, min_ratings=50),
            f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
        },
        tr, te, k=10, threshold=4
    )
    robustness_prec[fraction] = res["precision@10"]
    robustness_ndcg[fraction] = res["ndcg@10"]

robustness_prec = pd.DataFrame(robustness_prec)
robustness_prec.columns.name = "test_fraction"
print("Precision@10 across hold-out fractions:")
display(robustness_prec.round(4))

robustness_ndcg = pd.DataFrame(robustness_ndcg)
robustness_ndcg.columns.name = "test_fraction"
print("NDCG@10 across hold-out fractions:")
display(robustness_ndcg.round(4))

### Sensitivity to `min_ratings`

Evaluate how filtering rare movies via `min_ratings` affects accuracy and catalog coverage for both `PopularityRecommender` and `HybridRecommender`.

In [ ]:
min_ratings_grid = [10, 25, 50, 100, 200]
sens_models = {}
for mr in min_ratings_grid:
    sens_models[f"Popularity (min={mr})"] = PopularityRecommender(movies, min_ratings=mr)
    sens_models[f"Hybrid (alpha={best_alpha}, min={mr})"] = HybridRecommender(movies, alpha=best_alpha, min_ratings=mr)

sens_df = compare_models(sens_models, train, test, k=10, threshold=4)
display(sens_df[["precision@10", "ndcg@10", "hit_rate@10", "coverage", "personalization"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, prefix, color in [("Popularity", "Popularity", "darkorange"), (f"Hybrid (alpha={best_alpha})", "Hybrid", "seagreen")]:
    sub = sens_df.loc[[idx for idx in sens_df.index if idx.startswith(prefix)]]
    axes[0].plot(min_ratings_grid, sub["precision@10"], marker="o", label=label, color=color)
    axes[1].plot(min_ratings_grid, sub["coverage"], marker="o", label=label, color=color)

axes[0].set_title("Precision@10 vs. min_ratings")
axes[0].set_xlabel("min_ratings")
axes[0].set_ylabel("precision@10")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Catalog Coverage vs. min_ratings")
axes[1].set_xlabel("min_ratings")
axes[1].set_ylabel("coverage")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Performance by User Activity Level

Break down evaluation metrics across Light, Medium, and Heavy training-history tiers (equal-sized user tertiles) to examine how user history length affects personalized vs. popularity-based recommendations.

In [ ]:
segment_models = {
    "Random (floor)": RandomRecommender(movies, seed=0),
    "Content-based": ContentBasedRecommender(movies),
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
}
activity_df = evaluate_by_user_activity(segment_models, train, test, k=10, threshold=4, quantiles=3)
display(activity_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric in zip(axes, ["precision@10", "ndcg@10"]):
    pivot = activity_df.pivot(index="segment", columns="model", values=metric).loc[["Light", "Medium", "Heavy"]]
    pivot[list(segment_models.keys())].plot.bar(ax=ax)
    ax.set_title(f"{metric} by User Activity Tier")
    ax.set_xlabel("User Activity Tier")
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### Qualitative Comparison Across Contrasting Users

Inspect top-10 recommendations side-by-side for three contrasting users from MovieLens 100K: (1) a concentrated-genre fan, (2) a high-activity diverse-taste user, and (3) a light-history user.

In [ ]:
profiles = [user_genre_profile(ratings, movies, int(uid), threshold=4) for uid in sorted(ratings["user_id"].unique())]
prof_df = pd.DataFrame([
    {
        "user_id": p["user_id"],
        "total_ratings": p["total_ratings"],
        "positive_ratings": p["positive_ratings"],
        "n_genres_liked": len(p["genre_counts"]),
        "top_genre": p["genre_counts"].index[0] if not p["genre_counts"].empty else "None",
        "top_genre_share": (p["genre_counts"].iloc[0] / p["positive_ratings"]) if p["positive_ratings"] > 0 else 0.0,
    }
    for p in profiles
])

concentrated_uid = int(prof_df[prof_df["positive_ratings"] >= 20].sort_values(["top_genre_share", "user_id"], ascending=[False, True]).iloc[0]["user_id"])
diverse_uid = int(prof_df[prof_df["total_ratings"] >= 200].sort_values(["n_genres_liked", "total_ratings", "user_id"], ascending=[False, False, True]).iloc[0]["user_id"])
light_uid = int(prof_df[(prof_df["total_ratings"] == prof_df["total_ratings"].min()) & (prof_df["positive_ratings"] >= 8)].sort_values("user_id").iloc[0]["user_id"])

qual_models = {
    "Popularity": PopularityRecommender(movies, min_ratings=50),
    "Content-based": ContentBasedRecommender(movies),
    f"Hybrid (alpha={best_alpha})": HybridRecommender(movies, alpha=best_alpha, min_ratings=50),
}

for label, uid in [("Concentrated-genre user", concentrated_uid), ("Diverse heavy user", diverse_uid), ("Light-activity user", light_uid)]:
    p = user_genre_profile(ratings, movies, uid, threshold=4)
    top_g = ", ".join(f"{g} ({c})" for g, c in p["genre_counts"].head(5).items())
    print(f"=== {label}: User {uid} ({p['total_ratings']} ratings, {p['positive_ratings']} positive, mean={p['mean_rating']:.2f}) ===")
    print(f"Top liked genres: {top_g}")
    table = compare_user_recommendations(qual_models, ratings, movies, user_id=uid, n=10, threshold=4)
    display(table)
    pop_set = set(table["Popularity"])
    cb_set = set(table["Content-based"])
    hy_set = set(table[f"Hybrid (alpha={best_alpha})"])
    print(
        f"Top-10 overlap -> Content vs Popularity: {len(cb_set & pop_set)}/10 | "
        f"Hybrid vs Popularity: {len(hy_set & pop_set)}/10 | "
        f"Hybrid vs Content: {len(hy_set & cb_set)}/10\n"
    )

### Results

**Model comparison.** [Which model had the highest precision@10, recall@10, hit_rate@10, ndcg@10, and mrr@10? Give exact numbers and compare against the random floor.]

**Coverage & Personalization.** [Which model recommended the widest share of the catalog and highest personalization, and which the narrowest? What does that trade-off mean for users?]

**Choosing alpha.** [Which alpha gave the best precision@10? Describe how accuracy, ranking, coverage, and personalization changed as alpha went from 0 to 1.]

**Robustness.** [Did the ranking of the models stay the same across test_fraction 0.1, 0.2, and 0.3 for precision@10 and ndcg@10?]

**Sensitivity & User Segments.** [Summarize how min_ratings and user activity tiers affected Popularity, Content-based, and Hybrid models.]

**Qualitative Takeaways.** [Summarize what the side-by-side recommendations for the concentrated, diverse, and light users show about overlap and personalization.]

**Hypothesis.** [Evaluate whether the original hypothesis was supported or refined by the observed numbers.]